# 분리수거 판별기 — CNN 전이학습 (Colab)

EfficientNet-B0 전이학습으로 쓰레기 재질 4클래스(plastic/can/glass/paper) 분류 모델을 학습한다.

**사용법**
1. 로컬에서 `data/trashnet/` 폴더를 zip으로 압축: `trashnet.zip` (내부 구조: `trashnet/{plastic,can,glass,paper}/*.jpg`)
2. Colab 런타임 유형을 **GPU(T4)** 로 변경
3. 아래 업로드 셀에서 zip 업로드 후 전체 실행
4. 학습 완료 후 `best_model.pt` 다운로드 → 로컬 `model/best_model.pt`에 배치

데이터셋 출처: TrashNet (Gary Thung & Mindy Yang, https://github.com/garythung/trashnet)

In [ ]:
# 1. 데이터 업로드 및 압축 해제
from google.colab import files
import zipfile, os

uploaded = files.upload()  # trashnet.zip 업로드
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('data')
DATA_DIR = 'data/trashnet'
print(os.listdir(DATA_DIR))

In [ ]:
# 2. 데이터셋 구성 (train/val 8:2 분할, 증강 적용)
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

IMG_SIZE = 224
BATCH = 32
SEED = 42

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full = datasets.ImageFolder(DATA_DIR)
CLASS_NAMES = full.classes  # ['can', 'glass', 'paper', 'plastic'] (알파벳순)
print('클래스:', CLASS_NAMES)

n_val = int(len(full) * 0.2)
g = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(len(full)), [len(full) - n_val, n_val], generator=g)

train_ds = torch.utils.data.Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx.indices)
val_ds = torch.utils.data.Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf), val_idx.indices)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH, num_workers=2)
print(f'train {len(train_ds)} / val {len(val_ds)}')

In [ ]:
# 3. 클래스 불균형 가중치 (paper가 타 클래스의 약 2배)
from collections import Counter

label_counts = Counter(full.targets[i] for i in train_idx.indices)
total = sum(label_counts.values())
class_weights = torch.tensor(
    [total / (len(CLASS_NAMES) * label_counts[i]) for i in range(len(CLASS_NAMES))],
    dtype=torch.float,
)
print(dict(zip(CLASS_NAMES, class_weights.tolist())))

In [ ]:
# 4. 모델: EfficientNet-B0 전이학습 (분류 헤드만 교체)
from torchvision import models
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASS_NAMES))
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

In [ ]:
# 5. 학습 루프 (best val accuracy 체크포인트 저장 — Colab 세션 끊김 대비)
EPOCHS = 15
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    scheduler.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    val_acc = correct / len(val_ds)

    print(f'epoch {epoch+1:02d} | train_loss {train_loss/len(train_ds):.4f} | val_acc {val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'state_dict': model.state_dict(), 'classes': CLASS_NAMES}, 'best_model.pt')
        print(f'  -> best 갱신, 저장 (val_acc {best_acc:.4f})')

In [ ]:
# 6. 평가: 클래스별 리포트 + 혼동행렬
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

ckpt = torch.load('best_model.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_dl:
        y_true.extend(y.tolist())
        y_pred.extend(model(x.to(device)).argmax(1).cpu().tolist())

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
print(confusion_matrix(y_true, y_pred))

In [ ]:
# 7. 가중치 다운로드 → 로컬 model/best_model.pt 에 배치
from google.colab import files
files.download('best_model.pt')